### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="diamonds",
    dataset_year="2015",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://github.com/tidyverse/ggplot2/blob/main/data-raw/diamonds.csv",
    download_description="""
We download the CSV directly from the ggplot2 GitHub repository.

mkdir -p local-data-warehouse/diamonds/ && wget -O local-data-warehouse/diamonds/original_diamonds.csv https://raw.githubusercontent.com/tidyverse/ggplot2/main/data-raw/diamonds.csv
""",
    # References
    academic_reference_bibtex="""@incollection{wickham2016data,
  title={Data analysis},
  author={Wickham, Hadley},
  booktitle={ggplot2: Elegant graphics for data analysis},
  pages={189--201},
  year={2016},
  publisher={Springer}
}
""",
    academic_reference_bibtex_key="wickham2016data",
    license="MIT License",
    data_tags=["IID"],
    curation_comments="""
- Unlike TabArena, we log scale the target as it is a price and hence log scaling is generally recommended.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="price",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv(f"{dataset_mold.path}/original_diamonds.csv")

cat_features = [
    "cut",
    "color",
    "clarity",
]

# Shows a distribution shift based on the original order (which is gone after random
# shuffling) because the data is sorted by price
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")
df[task_mold.target_column_name] = np.log(df[task_mold.target_column_name])

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 53,940
Columns: 10
Use sampling: False (sample size: 53,940)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['x', 'y', 'z', 'carat', 'depth', 'table', 'clarity', 'color', 'cut']
Rows remaining as candidates after top-9 filter: 685 (of 53,940)

#### Duplicate Report
Total duplicate rows: 146 (0.27% of dataset)
Duplicate rows ignoring target: 345 (0.64% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.24,Ideal,G,VVS1,62.1,56.0,6.326149,3.97,4.00,2.47
1,0.58,Very Good,F,VVS2,60.0,57.0,7.696667,5.44,5.42,3.26
2,0.40,Ideal,E,VVS2,62.1,55.0,7.121252,4.76,4.74,2.95
3,0.43,Premium,E,VVS2,60.8,57.0,7.173192,4.92,4.89,2.98
4,1.55,Ideal,E,SI2,62.3,55.0,8.839422,7.44,7.37,4.61


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,cut,category,0.0,0.0,5.0,"Ideal, Premium, Very Good, Good, Fair"
1,color,category,0.0,0.0,7.0,"G, E, F, H, D, I, J"
2,clarity,category,0.0,0.0,8.0,"SI1, VS2, SI2, VS1, VVS2, VVS1, IF, I1"
3,carat,float64,0.0,0.0,273.0,"0.3, 0.31, 1.01, 0.7, 0.32, 1.0, 0.9, 0.41, 0.4, 0.71"
4,depth,float64,0.0,0.0,184.0,"62.0, 61.9, 61.8, 62.2, 62.1, 61.6, 62.3, 61.7, 62.4, 61.5"
5,table,float64,0.0,0.0,127.0,"56.0, 57.0, 58.0, 59.0, 55.0, 60.0, 54.0, 61.0, 62.0, 63.0"
6,price,float64,0.0,0.0,11602.0,"6.4052, 6.6871, 6.4378, 6.719, 6.6542, 6.6708, 6.5482, 6.2989, 6.5013, 6.3135"
7,x,float64,0.0,0.0,554.0,"4.37, 4.34, 4.33, 4.38, 4.32, 4.35, 4.39, 4.31, 4.36, 4.4"
8,y,float64,0.0,0.0,552.0,"4.34, 4.37, 4.35, 4.33, 4.32, 4.39, 4.38, 4.4, 4.31, 4.41"
9,z,float64,0.0,0.0,375.0,"2.7, 2.69, 2.71, 2.68, 2.72, 2.67, 2.73, 2.66, 2.74, 4.02"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
carat,53940.0,0.797940,0.474011,0.200000,5.010000
depth,53940.0,61.749405,1.432621,43.000000,79.000000
table,53940.0,57.457184,2.234491,43.000000,95.000000
price,53940.0,7.786768,1.014649,5.786897,9.842835
x,53940.0,5.731157,1.121761,0.000000,10.740000
y,53940.0,5.734526,1.142135,0.000000,58.900000
z,53940.0,3.538734,0.705699,0.000000,31.800000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column  rank                         
clarity 1           SI1  13065  24.22
        2           VS2  12258  22.73
        3           SI2   9194  17.04
        4           VS1   8171  15.15
        5          VVS2   5066   9.39
color   1             G  11292  20.93
        2             E   9797  18.16
        3             F   9542  17.69
        4             H   8304  15.39
        5             D   6775  12.56
cut     1         Ideal  21551  39.95
        2       Premium  13791  25.57
        3     Very Good  12082  22.40
        4          Good   4906   9.10
        5          Fair   1610   2.98

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.115,-0.056,1.03,0.017,log,494817.1,163681.9,lognormal


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to diamonds/019d7367-779c-707e-b777-849180166b74


019d7367-779c-707e-b777-849180166b74
b151949b12a5563f61b3915e8915ade6e09db41cf19e145d4545d749d68f1685
